# Notebook 2: Advanced Graph Queries

This notebook is for the point where basic connectivity is no longer the question. The focus here is interpretation: understanding dependency hot spots, inheritance shape, import behavior, and documentation coverage inside a populated graph.

Each section highlights a different angle of engineering value so the notebook reads like an analysis session rather than a loose set of Cypher snippets.

In [ ]:
# Setup
import sys
from pathlib import Path
import os
from dotenv import load_dotenv
import pandas as pd

env_path = Path('..') / '.env'
load_dotenv(dotenv_path=env_path)
sys.path.append(str(Path('../src').resolve()))

from neo4j_agent import Neo4jAgent

# Connect to Neo4j
agent = Neo4jAgent(
    uri=os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    user=os.getenv('NEO4J_USER', 'neo4j'),
    password=os.getenv('NEO4J_PASSWORD', 'password')
)

status = agent.verify_connection()
print(f"Using environment file: {env_path}")
print(f"✅ Connected to Neo4j: {status['connected']}")

## 1. Dependency Analysis

Dependency queries are where the graph starts to become operationally useful. Instead of reading build files and imports by hand, you can surface classes that concentrate external coupling and identify where changes will ripple first.

In [ ]:
# Find classes with most dependencies
query = """
MATCH (c:Class)-[:DEPENDS_ON]->(dep:ExternalDependency)
WITH c, count(dep) AS DepCount, collect(dep.name) AS Dependencies
RETURN c.name AS Class, DepCount, Dependencies
ORDER BY DepCount DESC
LIMIT 10
"""

results = agent.query(query)
df = pd.DataFrame(results)
print("📦 Classes with Most Dependencies:\n")
print(df)

In [ ]:
# Find which classes use a specific dependency
dependency_name = "core-runtime"

query = """
MATCH (c:Class)-[:DEPENDS_ON]->(dep:ExternalDependency)
WHERE dep.name CONTAINS $dep_name
RETURN c.name AS Class, c.category AS Category
ORDER BY c.name
"""

results = agent.query(query, {'dep_name': dependency_name})
print(f"\n🔍 Classes using {dependency_name}:\n")
for record in results:
    print(f"  • {record['Class']} ({record['Category']})")

## 2. Inheritance Analysis

Inheritance depth often reveals framework pressure, extension strategy, and potential maintenance cost. These queries help you see where the object model is shallow and where it has accumulated layers.

In [ ]:
# Find inheritance depth (classes with most ancestors)
query = """
MATCH path = (child:Class)-[:EXTENDS*1..5]->(ancestor:Class)
WITH child, length(path) AS Depth, collect(ancestor.name) AS Ancestors
RETURN child.name AS Class, Depth, Ancestors
ORDER BY Depth DESC
LIMIT 5
"""

results = agent.query(query)
print("🌳 Deepest Inheritance Hierarchies:\n")
for record in results:
    print(f"  {record['Class']} (depth: {record['Depth']})")
    print(f"    → {' → '.join(record['Ancestors'])}\n")

In [ ]:
# Find most common base classes
query = """
MATCH (child:Class)-[:EXTENDS]->(parent:Class)
WITH parent, count(child) AS ChildCount, collect(child.name) AS Children
WHERE ChildCount > 1
RETURN parent.name AS BaseClass, ChildCount, Children
ORDER BY ChildCount DESC
"""

results = agent.query(query)
print("\n👪 Most Extended Base Classes:\n")
for record in results:
    print(f"  {record['BaseClass']} ({record['ChildCount']} children)")
    print(f"    Children: {', '.join(record['Children'][:5])}{'...' if len(record['Children']) > 5 else ''}\n")

## 3. Code Metrics

This section shifts from topology to volume. Method counts, empty classes, and visibility breakdowns give you a quick signal on whether the graph looks balanced or whether a few classes are carrying too much complexity.

In [ ]:
# Method count per class
query = """
MATCH (c:Class)
OPTIONAL MATCH (c)-[:DEFINES_METHOD]->(m:Method)
WITH c, count(m) AS MethodCount
RETURN c.name AS Class, c.category AS Category, MethodCount
ORDER BY MethodCount DESC
LIMIT 10
"""

results = agent.query(query)
df = pd.DataFrame(results)
print("📊 Classes with Most Methods:\n")
print(df)

print(f"\n📈 Average methods per class: {df['MethodCount'].mean():.1f}")

In [ ]:
# Find classes with no methods (interfaces or empty classes)
query = """
MATCH (c:Class)
WHERE NOT (c)-[:DEFINES_METHOD]->()
RETURN c.name AS Class, c.category AS Category, c.isAbstract AS IsAbstract
"""

results = agent.query(query)
print("\n🔍 Classes with No Methods:\n")
for record in results:
    print(f"  • {record['Class']} ({record['Category']})" + 
          (" - Abstract" if record['IsAbstract'] else ""))

In [ ]:
# Method visibility distribution
query = """
MATCH (m:Method)
RETURN m.visibility AS Visibility, count(*) AS Count
ORDER BY Count DESC
"""

results = agent.query(query)
df = pd.DataFrame(results)
print("\n🔐 Method Visibility Distribution:\n")
print(df)

## 4. Import Analysis

Imports are a good proxy for coordination cost. If a class pulls in too many external symbols, it usually becomes harder to reason about, harder to test, and more brittle during version changes.

In [ ]:
# Most commonly imported packages
query = """
MATCH (c:Class)-[:IMPORTS]->(i:Import)
WITH i.path AS ImportPath, count(c) AS UsageCount
RETURN ImportPath, UsageCount
ORDER BY UsageCount DESC
LIMIT 10
"""

results = agent.query(query)
print("📚 Most Commonly Imported Packages:\n")
for i, record in enumerate(results, 1):
    print(f"  {i}. {record['ImportPath']} ({record['UsageCount']} classes)")

In [ ]:
# Find classes with most imports
query = """
MATCH (c:Class)-[:IMPORTS]->(i:Import)
WITH c, count(i) AS ImportCount
RETURN c.name AS Class, ImportCount
ORDER BY ImportCount DESC
LIMIT 5
"""

results = agent.query(query)
print("\n🔗 Classes with Most Imports:\n")
for record in results:
    print(f"  • {record['Class']}: {record['ImportCount']} imports")

## 5. Documentation Coverage

Graphs become more valuable when they connect code to supporting material. If you model documentation nodes, this section gives you a quick way to see whether the graph is only structural or whether it is also explainable.

In [ ]:
# Find classes with documentation
query = """
MATCH (md:MarkdownFile)-[:RESOLVES_TO]->(c:Class)
RETURN c.name AS Class, collect(md.path) AS Documentation
ORDER BY c.name
"""

results = agent.query(query)
print("📖 Classes with Documentation:\n")
for record in results:
    print(f"  • {record['Class']}")
    for doc in record['Documentation']:
        print(f"    → {doc}")

In [ ]:
# Documentation coverage statistics
query = """
MATCH (c:Class)
OPTIONAL MATCH (md:MarkdownFile)-[:RESOLVES_TO]->(c)
WITH c, count(md) AS DocCount
RETURN 
  count(c) AS TotalClasses,
  sum(CASE WHEN DocCount > 0 THEN 1 ELSE 0 END) AS DocumentedClasses,
  round(100.0 * sum(CASE WHEN DocCount > 0 THEN 1 ELSE 0 END) / count(c), 1) AS CoveragePercent
"""

results = agent.query(query)
stats = results[0]
print("\n📊 Documentation Coverage:\n")
print(f"  Total Classes: {stats['TotalClasses']}")
print(f"  Documented Classes: {stats['DocumentedClasses']}")
print(f"  Coverage: {stats['CoveragePercent']}%")

## 6. Version Comparison

Version-aware graph queries are useful when one branch of the system is evolving faster than another. They let you ask whether a class is stable, duplicated across versions, or drifting in ways that should change how generation prompts are built.

In [ ]:
# Find classes present in multiple versions
query = """
MATCH (c:Class)
WITH c.qualifiedName AS ClassName, collect(DISTINCT c.gitTag) AS Versions
WHERE size(Versions) > 1
RETURN ClassName, Versions
ORDER BY ClassName
"""

results = agent.query(query)
print("🔄 Classes in Multiple Versions:\n")
for record in results:
    print(f"  • {record['ClassName']}")
    print(f"    Versions: {', '.join(record['Versions'])}\n")

In [ ]:
# Version statistics
query = """
MATCH (c:Class)
WITH c.gitTag AS Version, count(c) AS ClassCount
RETURN Version, ClassCount
ORDER BY Version
"""

results = agent.query(query)
df = pd.DataFrame(results)
print("\n📈 Classes per Version:\n")
print(df)

## 7. Custom Analysis

The final section is deliberately open-ended. Once the base patterns are clear, this is where you test the questions that matter to your codebase rather than the canned ones that mattered to the sample.

In [ ]:
# Your custom query
custom_query = """
// Example: Find test method classes
MATCH (c:Class)
WHERE c.category = 'TestMethod'
RETURN c.name, c.gitTag
ORDER BY c.name
"""

results = agent.query(custom_query)
print("🔬 Custom Query Results:\n")
for record in results:
    print(f"  {record}")

## Cleanup

Closing the session keeps the notebook repeatable. It is a small step, but it avoids leaving connection state behind while you move on to the next analysis notebook.

In [ ]:
agent.close()
print("✅ Neo4j connection closed")

## Next Steps

A useful way to continue from here is to promote your best exploratory queries into repeatable tooling. Once a query becomes part of a workflow, it belongs either in code, in an MCP tool definition, or in a dashboard rather than only in a notebook.

Useful references:

- [README.md](../README.md) for the repository overview.
- [ARCHITECTURE.md](../docs/ARCHITECTURE.md) for the analysis model.
- [MCP_INTEGRATION.md](../docs/MCP_INTEGRATION.md) for turning stable queries into editor-accessible tools.
- [Neo4j Graph Data Science](https://neo4j.com/docs/graph-data-science/current/) for deeper graph algorithms.